# SAP O2C Data Modelling

Schemas are sourced from `docs/sap_o2c_schemas.md` (19 tables, validated against `data/sap-o2c-data/`).


In [ ]:
import pandas as pd
import duckdb
import os
import glob
import magic_duckdb

DATA_DIR = '../data/sap-o2c-data'

con = duckdb.connect('o2c_data.db')

# magic_duckdb setup
magic_duckdb.MAGIC_NAME = 'sql'
%load_ext magic_duckdb
%sql -co con
%config MagicDuckDB.displaylimit = 0

## Schema Creation (DDL)
Creating all 19 tables with proper types from `sap_o2c_schemas.md`.


In [ ]:
ddl = '''
-- Drop and recreate all tables with proper DDL from sap_o2c_schemas.md

CREATE TABLE IF NOT EXISTS billing_document_headers (
    billing_document           VARCHAR(10)    NOT NULL PRIMARY KEY,
    billing_document_type      VARCHAR(4),
    creation_date              DATE,
    creation_time              TIME,
    last_change_datetime       TIMESTAMPTZ,
    billing_document_date      DATE,
    billing_document_is_cancelled BOOLEAN     DEFAULT FALSE,
    cancelled_billing_document VARCHAR(10),
    total_net_amount           NUMERIC(15, 2),
    transaction_currency       VARCHAR(5),
    company_code               VARCHAR(4),
    fiscal_year                VARCHAR(4),
    accounting_document        VARCHAR(10),
    sold_to_party              VARCHAR(10)
);

CREATE TABLE IF NOT EXISTS billing_document_cancellations (
    billing_document              VARCHAR(10)    NOT NULL PRIMARY KEY,
    billing_document_type         VARCHAR(4),
    creation_date                 DATE,
    creation_time                 TIME,
    last_change_datetime          TIMESTAMPTZ,
    billing_document_date         DATE,
    billing_document_is_cancelled BOOLEAN        DEFAULT FALSE,
    cancelled_billing_document    VARCHAR(10),
    total_net_amount              NUMERIC(15, 2),
    transaction_currency          VARCHAR(5),
    company_code                  VARCHAR(4),
    fiscal_year                   VARCHAR(4),
    accounting_document           VARCHAR(10),
    sold_to_party                 VARCHAR(10)
);

CREATE TABLE IF NOT EXISTS billing_document_items (
    billing_document          VARCHAR(10)   NOT NULL,
    billing_document_item     VARCHAR(6)    NOT NULL,
    material                  VARCHAR(40),
    billing_quantity          NUMERIC(15, 3),
    billing_quantity_unit     VARCHAR(3),
    net_amount                NUMERIC(15, 2),
    transaction_currency      VARCHAR(5),
    reference_sd_document     VARCHAR(10),
    reference_sd_document_item VARCHAR(6),
    PRIMARY KEY (billing_document, billing_document_item)
);

CREATE TABLE IF NOT EXISTS business_partner_addresses (
    business_partner           VARCHAR(10)   NOT NULL,
    address_id                 VARCHAR(10)   NOT NULL,
    validity_start_date        DATE,
    validity_end_date          DATE,
    address_uuid               VARCHAR(36),
    address_time_zone          VARCHAR(10),
    city_name                  VARCHAR(40),
    country                    VARCHAR(3),
    po_box                     VARCHAR(10),
    po_box_deviating_city_name VARCHAR(40),
    po_box_deviating_country   VARCHAR(3),
    po_box_deviating_region    VARCHAR(3),
    po_box_is_without_number   BOOLEAN       DEFAULT FALSE,
    po_box_lobby_name          VARCHAR(40),
    po_box_postal_code         VARCHAR(10),
    postal_code                VARCHAR(10),
    region                     VARCHAR(3),
    street_name                VARCHAR(60),
    tax_jurisdiction           VARCHAR(15),
    transport_zone             VARCHAR(10),
    PRIMARY KEY (business_partner, address_id)
);

CREATE TABLE IF NOT EXISTS business_partners (
    business_partner           VARCHAR(10)   NOT NULL PRIMARY KEY,
    customer                   VARCHAR(10),
    business_partner_category  VARCHAR(1),
    business_partner_full_name VARCHAR(81),
    business_partner_grouping  VARCHAR(4),
    business_partner_name      VARCHAR(81),
    correspondence_language    VARCHAR(2),
    created_by_user            VARCHAR(12),
    creation_date              DATE,
    creation_time              TIME,
    first_name                 VARCHAR(40),
    form_of_address            VARCHAR(4),
    industry                   VARCHAR(10),
    last_change_date           DATE,
    last_name                  VARCHAR(40),
    organization_bp_name1      VARCHAR(40),
    organization_bp_name2      VARCHAR(40),
    business_partner_is_blocked BOOLEAN      DEFAULT FALSE,
    is_marked_for_archiving    BOOLEAN       DEFAULT FALSE
);

CREATE TABLE IF NOT EXISTS customer_company_assignments (
    customer                         VARCHAR(10)   NOT NULL,
    company_code                     VARCHAR(4)    NOT NULL,
    accounting_clerk                 VARCHAR(2),
    accounting_clerk_fax_number      VARCHAR(31),
    accounting_clerk_internet_address VARCHAR(130),
    accounting_clerk_phone_number    VARCHAR(30),
    alternative_payer_account        VARCHAR(10),
    payment_blocking_reason          VARCHAR(1),
    payment_methods_list             VARCHAR(10),
    payment_terms                    VARCHAR(4),
    reconciliation_account           VARCHAR(10),
    deletion_indicator               BOOLEAN       DEFAULT FALSE,
    customer_account_group           VARCHAR(4),
    PRIMARY KEY (customer, company_code)
);

CREATE TABLE IF NOT EXISTS customer_sales_area_assignments (
    customer                      VARCHAR(10)   NOT NULL,
    sales_organization            VARCHAR(4)    NOT NULL,
    distribution_channel          VARCHAR(2)    NOT NULL,
    division                      VARCHAR(2)    NOT NULL,
    billing_is_blocked_for_customer VARCHAR(2),
    complete_delivery_is_defined  BOOLEAN       DEFAULT FALSE,
    credit_control_area           VARCHAR(4),
    currency                      VARCHAR(5),
    customer_payment_terms        VARCHAR(4),
    delivery_priority             VARCHAR(2),
    incoterms_classification      VARCHAR(3),
    incoterms_location1           VARCHAR(70),
    sales_group                   VARCHAR(3),
    sales_office                  VARCHAR(4),
    shipping_condition            VARCHAR(2),
    sls_unlmtd_ovrdeliv_is_allwd  BOOLEAN       DEFAULT FALSE,
    supplying_plant               VARCHAR(4),
    sales_district                VARCHAR(6),
    exchange_rate_type            VARCHAR(4),
    PRIMARY KEY (customer, sales_organization, distribution_channel, division)
);

CREATE TABLE IF NOT EXISTS journal_entry_items_accounts_receivable (
    company_code                   VARCHAR(4)    NOT NULL,
    fiscal_year                    VARCHAR(4)    NOT NULL,
    accounting_document            VARCHAR(10)   NOT NULL,
    accounting_document_item       VARCHAR(3)    NOT NULL,
    gl_account                     VARCHAR(10),
    reference_document             VARCHAR(16),
    cost_center                    VARCHAR(10),
    profit_center                  VARCHAR(10),
    transaction_currency           VARCHAR(5),
    amount_in_transaction_currency NUMERIC(23, 2),
    company_code_currency          VARCHAR(5),
    amount_in_company_code_currency NUMERIC(23, 2),
    posting_date                   DATE,
    document_date                  DATE,
    accounting_document_type       VARCHAR(2),
    assignment_reference           VARCHAR(18),
    last_change_datetime           TIMESTAMPTZ,
    customer                       VARCHAR(10),
    financial_account_type         VARCHAR(1),
    clearing_date                  DATE,
    clearing_accounting_document   VARCHAR(10),
    clearing_doc_fiscal_year       VARCHAR(4),
    PRIMARY KEY (company_code, fiscal_year, accounting_document, accounting_document_item)
);

CREATE TABLE IF NOT EXISTS outbound_delivery_headers (
    delivery_document               VARCHAR(10)   NOT NULL PRIMARY KEY,
    actual_goods_movement_date      DATE,
    actual_goods_movement_time      TIME,
    creation_date                   DATE,
    creation_time                   TIME,
    delivery_block_reason           VARCHAR(2),
    hdr_general_incompletion_status VARCHAR(1),
    header_billing_block_reason     VARCHAR(2),
    last_change_date                DATE,
    overall_goods_movement_status   VARCHAR(1),
    overall_picking_status          VARCHAR(1),
    overall_proof_of_delivery_status VARCHAR(1),
    shipping_point                  VARCHAR(4)
);

CREATE TABLE IF NOT EXISTS outbound_delivery_items (
    delivery_document          VARCHAR(10)   NOT NULL,
    delivery_document_item     VARCHAR(6)    NOT NULL,
    actual_delivery_quantity   NUMERIC(15, 3),
    batch                      VARCHAR(10),
    delivery_quantity_unit     VARCHAR(3),
    item_billing_block_reason  VARCHAR(2),
    last_change_date           DATE,
    plant                      VARCHAR(4),
    reference_sd_document      VARCHAR(10),
    reference_sd_document_item VARCHAR(6),
    storage_location           VARCHAR(4),
    PRIMARY KEY (delivery_document, delivery_document_item)
);

CREATE TABLE IF NOT EXISTS payments_accounts_receivable (
    company_code                    VARCHAR(4)    NOT NULL,
    fiscal_year                     VARCHAR(4)    NOT NULL,
    accounting_document             VARCHAR(10)   NOT NULL,
    accounting_document_item        VARCHAR(3)    NOT NULL,
    clearing_date                   DATE,
    clearing_accounting_document    VARCHAR(10),
    clearing_doc_fiscal_year        VARCHAR(4),
    amount_in_transaction_currency  NUMERIC(23, 2),
    transaction_currency            VARCHAR(5),
    amount_in_company_code_currency NUMERIC(23, 2),
    company_code_currency           VARCHAR(5),
    customer                        VARCHAR(10),
    invoice_reference               VARCHAR(10),
    invoice_reference_fiscal_year   VARCHAR(4),
    sales_document                  VARCHAR(10),
    sales_document_item             VARCHAR(6),
    posting_date                    DATE,
    document_date                   DATE,
    assignment_reference            VARCHAR(18),
    gl_account                      VARCHAR(10),
    financial_account_type          VARCHAR(1),
    profit_center                   VARCHAR(10),
    cost_center                     VARCHAR(10),
    PRIMARY KEY (company_code, fiscal_year, accounting_document, accounting_document_item)
);

CREATE TABLE IF NOT EXISTS plants (
    plant                             VARCHAR(4)    NOT NULL PRIMARY KEY,
    plant_name                        VARCHAR(30),
    valuation_area                    VARCHAR(4),
    plant_customer                    VARCHAR(10),
    plant_supplier                    VARCHAR(10),
    factory_calendar                  VARCHAR(2),
    default_purchasing_organization   VARCHAR(4),
    sales_organization                VARCHAR(4),
    address_id                        VARCHAR(10),
    plant_category                    VARCHAR(1),
    distribution_channel              VARCHAR(2),
    division                          VARCHAR(2),
    language                          VARCHAR(2),
    is_marked_for_archiving           BOOLEAN       DEFAULT FALSE
);

CREATE TABLE IF NOT EXISTS product_descriptions (
    product             VARCHAR(40)   NOT NULL,
    language            VARCHAR(2)    NOT NULL,
    product_description VARCHAR(40),
    PRIMARY KEY (product, language)
);

CREATE TABLE IF NOT EXISTS product_plants (
    product                      VARCHAR(40)   NOT NULL,
    plant                        VARCHAR(4)    NOT NULL,
    country_of_origin            VARCHAR(3),
    region_of_origin             VARCHAR(3),
    production_invtry_managed_loc VARCHAR(4),
    availability_check_type      VARCHAR(2),
    fiscal_year_variant          VARCHAR(2),
    profit_center                VARCHAR(10),
    mrp_type                     VARCHAR(2),
    PRIMARY KEY (product, plant)
);

CREATE TABLE IF NOT EXISTS product_storage_locations (
    product                               VARCHAR(40)   NOT NULL,
    plant                                 VARCHAR(4)    NOT NULL,
    storage_location                      VARCHAR(4)    NOT NULL,
    physical_inventory_block_ind          VARCHAR(1),
    date_of_last_posted_cnt_un_rstrcd_stk DATE,
    PRIMARY KEY (product, plant, storage_location)
);

CREATE TABLE IF NOT EXISTS products (
    product                          VARCHAR(40)   NOT NULL PRIMARY KEY,
    product_type                     VARCHAR(4),
    cross_plant_status               VARCHAR(2),
    cross_plant_status_validity_date DATE,
    creation_date                    DATE,
    created_by_user                  VARCHAR(12),
    last_change_date                 DATE,
    last_change_datetime             TIMESTAMPTZ,
    is_marked_for_deletion           BOOLEAN       DEFAULT FALSE,
    product_old_id                   VARCHAR(40),
    gross_weight                     NUMERIC(13, 3),
    weight_unit                      VARCHAR(3),
    net_weight                       NUMERIC(13, 3),
    product_group                    VARCHAR(9),
    base_unit                        VARCHAR(3),
    division                         VARCHAR(2),
    industry_sector                  VARCHAR(1)
);

CREATE TABLE IF NOT EXISTS sales_order_headers (
    sales_order                     VARCHAR(10)   NOT NULL PRIMARY KEY,
    sales_order_type                VARCHAR(4),
    sales_organization              VARCHAR(4),
    distribution_channel            VARCHAR(2),
    organization_division           VARCHAR(2),
    sales_group                     VARCHAR(3),
    sales_office                    VARCHAR(4),
    sold_to_party                   VARCHAR(10),
    creation_date                   DATE,
    created_by_user                 VARCHAR(12),
    last_change_datetime            TIMESTAMPTZ,
    total_net_amount                NUMERIC(15, 2),
    overall_delivery_status         VARCHAR(1),
    overall_ord_reltd_billg_status  VARCHAR(1),
    overall_sd_doc_reference_status VARCHAR(1),
    transaction_currency            VARCHAR(5),
    pricing_date                    DATE,
    requested_delivery_date         DATE,
    header_billing_block_reason     VARCHAR(2),
    delivery_block_reason           VARCHAR(2),
    incoterms_classification        VARCHAR(3),
    incoterms_location1             VARCHAR(70),
    customer_payment_terms          VARCHAR(4),
    total_credit_check_status       VARCHAR(1)
);

CREATE TABLE IF NOT EXISTS sales_order_items (
    sales_order                VARCHAR(10)   NOT NULL,
    sales_order_item           VARCHAR(6)    NOT NULL,
    sales_order_item_category  VARCHAR(4),
    material                   VARCHAR(40),
    requested_quantity         NUMERIC(15, 3),
    requested_quantity_unit    VARCHAR(3),
    transaction_currency       VARCHAR(5),
    net_amount                 NUMERIC(15, 2),
    material_group             VARCHAR(9),
    production_plant           VARCHAR(4),
    storage_location           VARCHAR(4),
    sales_document_rjcn_reason VARCHAR(2),
    item_billing_block_reason  VARCHAR(2),
    PRIMARY KEY (sales_order, sales_order_item)
);

CREATE TABLE IF NOT EXISTS sales_order_schedule_lines (
    sales_order                           VARCHAR(10)   NOT NULL,
    sales_order_item                      VARCHAR(6)    NOT NULL,
    schedule_line                         VARCHAR(4)    NOT NULL,
    confirmed_delivery_date               DATE,
    order_quantity_unit                   VARCHAR(3),
    confd_order_qty_by_matl_avail_check   NUMERIC(15, 3),
    PRIMARY KEY (sales_order, sales_order_item, schedule_line)
);
'''

# Execute each statement individually
for stmt in [s.strip() for s in ddl.split(';') if s.strip()]:
    con.execute(stmt)

tables = con.execute("SELECT table_name FROM duckdb_tables() ORDER BY table_name").fetchdf()
print(f'Created {len(tables)} tables:')
print(tables['table_name'].tolist())

## Data Loading
Loads all JSONL files using explicit camelCase→snake_case column mapping.
All values are read as VARCHAR first, then cast via `NULLIF(TRIM(...), '')` to handle empty strings and NULLs cleanly.
Nested `creationTime` / `actualGoodsMovementTime` objects are excluded from initial load (DuckDB reads the object as-is).


In [ ]:
import os
import glob

DATA_DIR = '../data/sap-o2c-data'

TABLE_COLUMN_MAPS = {'billing_document_headers': {'billingDocument': 'billing_document', 'billingDocumentType': 'billing_document_type', 'creationDate': 'creation_date', 'creationTime': 'creation_time', 'lastChangeDateTime': 'last_change_datetime', 'billingDocumentDate': 'billing_document_date', 'billingDocumentIsCancelled': 'billing_document_is_cancelled', 'cancelledBillingDocument': 'cancelled_billing_document', 'totalNetAmount': 'total_net_amount', 'transactionCurrency': 'transaction_currency', 'companyCode': 'company_code', 'fiscalYear': 'fiscal_year', 'accountingDocument': 'accounting_document', 'soldToParty': 'sold_to_party'}, 'billing_document_cancellations': {'billingDocument': 'billing_document', 'billingDocumentType': 'billing_document_type', 'creationDate': 'creation_date', 'creationTime': 'creation_time', 'lastChangeDateTime': 'last_change_datetime', 'billingDocumentDate': 'billing_document_date', 'billingDocumentIsCancelled': 'billing_document_is_cancelled', 'cancelledBillingDocument': 'cancelled_billing_document', 'totalNetAmount': 'total_net_amount', 'transactionCurrency': 'transaction_currency', 'companyCode': 'company_code', 'fiscalYear': 'fiscal_year', 'accountingDocument': 'accounting_document', 'soldToParty': 'sold_to_party'}, 'billing_document_items': {'billingDocument': 'billing_document', 'billingDocumentItem': 'billing_document_item', 'material': 'material', 'billingQuantity': 'billing_quantity', 'billingQuantityUnit': 'billing_quantity_unit', 'netAmount': 'net_amount', 'transactionCurrency': 'transaction_currency', 'referenceSdDocument': 'reference_sd_document', 'referenceSdDocumentItem': 'reference_sd_document_item'}, 'business_partner_addresses': {'businessPartner': 'business_partner', 'addressId': 'address_id', 'validityStartDate': 'validity_start_date', 'validityEndDate': 'validity_end_date', 'addressUuid': 'address_uuid', 'addressTimeZone': 'address_time_zone', 'cityName': 'city_name', 'country': 'country', 'poBox': 'po_box', 'poBoxDeviatingCityName': 'po_box_deviating_city_name', 'poBoxDeviatingCountry': 'po_box_deviating_country', 'poBoxDeviatingRegion': 'po_box_deviating_region', 'poBoxIsWithoutNumber': 'po_box_is_without_number', 'poBoxLobbyName': 'po_box_lobby_name', 'poBoxPostalCode': 'po_box_postal_code', 'postalCode': 'postal_code', 'region': 'region', 'streetName': 'street_name', 'taxJurisdiction': 'tax_jurisdiction', 'transportZone': 'transport_zone'}, 'business_partners': {'businessPartner': 'business_partner', 'customer': 'customer', 'businessPartnerCategory': 'business_partner_category', 'businessPartnerFullName': 'business_partner_full_name', 'businessPartnerGrouping': 'business_partner_grouping', 'businessPartnerName': 'business_partner_name', 'correspondenceLanguage': 'correspondence_language', 'createdByUser': 'created_by_user', 'creationDate': 'creation_date', 'creationTime': 'creation_time', 'firstName': 'first_name', 'formOfAddress': 'form_of_address', 'industry': 'industry', 'lastChangeDate': 'last_change_date', 'lastName': 'last_name', 'organizationBpName1': 'organization_bp_name1', 'organizationBpName2': 'organization_bp_name2', 'businessPartnerIsBlocked': 'business_partner_is_blocked', 'isMarkedForArchiving': 'is_marked_for_archiving'}, 'customer_company_assignments': {'customer': 'customer', 'companyCode': 'company_code', 'accountingClerk': 'accounting_clerk', 'accountingClerkFaxNumber': 'accounting_clerk_fax_number', 'accountingClerkInternetAddress': 'accounting_clerk_internet_address', 'accountingClerkPhoneNumber': 'accounting_clerk_phone_number', 'alternativePayerAccount': 'alternative_payer_account', 'paymentBlockingReason': 'payment_blocking_reason', 'paymentMethodsList': 'payment_methods_list', 'paymentTerms': 'payment_terms', 'reconciliationAccount': 'reconciliation_account', 'deletionIndicator': 'deletion_indicator', 'customerAccountGroup': 'customer_account_group'}, 'customer_sales_area_assignments': {'customer': 'customer', 'salesOrganization': 'sales_organization', 'distributionChannel': 'distribution_channel', 'division': 'division', 'billingIsBlockedForCustomer': 'billing_is_blocked_for_customer', 'completeDeliveryIsDefined': 'complete_delivery_is_defined', 'creditControlArea': 'credit_control_area', 'currency': 'currency', 'customerPaymentTerms': 'customer_payment_terms', 'deliveryPriority': 'delivery_priority', 'incotermsClassification': 'incoterms_classification', 'incotermsLocation1': 'incoterms_location1', 'salesGroup': 'sales_group', 'salesOffice': 'sales_office', 'shippingCondition': 'shipping_condition', 'slsUnlmtdOvrdelivIsAllwd': 'sls_unlmtd_ovrdeliv_is_allwd', 'supplyingPlant': 'supplying_plant', 'salesDistrict': 'sales_district', 'exchangeRateType': 'exchange_rate_type'}, 'journal_entry_items_accounts_receivable': {'companyCode': 'company_code', 'fiscalYear': 'fiscal_year', 'accountingDocument': 'accounting_document', 'glAccount': 'gl_account', 'referenceDocument': 'reference_document', 'costCenter': 'cost_center', 'profitCenter': 'profit_center', 'transactionCurrency': 'transaction_currency', 'amountInTransactionCurrency': 'amount_in_transaction_currency', 'companyCodeCurrency': 'company_code_currency', 'amountInCompanyCodeCurrency': 'amount_in_company_code_currency', 'postingDate': 'posting_date', 'documentDate': 'document_date', 'accountingDocumentType': 'accounting_document_type', 'accountingDocumentItem': 'accounting_document_item', 'assignmentReference': 'assignment_reference', 'lastChangeDateTime': 'last_change_datetime', 'customer': 'customer', 'financialAccountType': 'financial_account_type', 'clearingDate': 'clearing_date', 'clearingAccountingDocument': 'clearing_accounting_document', 'clearingDocFiscalYear': 'clearing_doc_fiscal_year'}, 'outbound_delivery_headers': {'actualGoodsMovementDate': 'actual_goods_movement_date', 'actualGoodsMovementTime': 'actual_goods_movement_time', 'creationDate': 'creation_date', 'creationTime': 'creation_time', 'deliveryBlockReason': 'delivery_block_reason', 'deliveryDocument': 'delivery_document', 'hdrGeneralIncompletionStatus': 'hdr_general_incompletion_status', 'headerBillingBlockReason': 'header_billing_block_reason', 'lastChangeDate': 'last_change_date', 'overallGoodsMovementStatus': 'overall_goods_movement_status', 'overallPickingStatus': 'overall_picking_status', 'overallProofOfDeliveryStatus': 'overall_proof_of_delivery_status', 'shippingPoint': 'shipping_point'}, 'outbound_delivery_items': {'actualDeliveryQuantity': 'actual_delivery_quantity', 'batch': 'batch', 'deliveryDocument': 'delivery_document', 'deliveryDocumentItem': 'delivery_document_item', 'deliveryQuantityUnit': 'delivery_quantity_unit', 'itemBillingBlockReason': 'item_billing_block_reason', 'lastChangeDate': 'last_change_date', 'plant': 'plant', 'referenceSdDocument': 'reference_sd_document', 'referenceSdDocumentItem': 'reference_sd_document_item', 'storageLocation': 'storage_location'}, 'payments_accounts_receivable': {'companyCode': 'company_code', 'fiscalYear': 'fiscal_year', 'accountingDocument': 'accounting_document', 'accountingDocumentItem': 'accounting_document_item', 'clearingDate': 'clearing_date', 'clearingAccountingDocument': 'clearing_accounting_document', 'clearingDocFiscalYear': 'clearing_doc_fiscal_year', 'amountInTransactionCurrency': 'amount_in_transaction_currency', 'transactionCurrency': 'transaction_currency', 'amountInCompanyCodeCurrency': 'amount_in_company_code_currency', 'companyCodeCurrency': 'company_code_currency', 'customer': 'customer', 'invoiceReference': 'invoice_reference', 'invoiceReferenceFiscalYear': 'invoice_reference_fiscal_year', 'salesDocument': 'sales_document', 'salesDocumentItem': 'sales_document_item', 'postingDate': 'posting_date', 'documentDate': 'document_date', 'assignmentReference': 'assignment_reference', 'glAccount': 'gl_account', 'financialAccountType': 'financial_account_type', 'profitCenter': 'profit_center', 'costCenter': 'cost_center'}, 'plants': {'plant': 'plant', 'plantName': 'plant_name', 'valuationArea': 'valuation_area', 'plantCustomer': 'plant_customer', 'plantSupplier': 'plant_supplier', 'factoryCalendar': 'factory_calendar', 'defaultPurchasingOrganization': 'default_purchasing_organization', 'salesOrganization': 'sales_organization', 'addressId': 'address_id', 'plantCategory': 'plant_category', 'distributionChannel': 'distribution_channel', 'division': 'division', 'language': 'language', 'isMarkedForArchiving': 'is_marked_for_archiving'}, 'product_descriptions': {'product': 'product', 'language': 'language', 'productDescription': 'product_description'}, 'product_plants': {'product': 'product', 'plant': 'plant', 'countryOfOrigin': 'country_of_origin', 'regionOfOrigin': 'region_of_origin', 'productionInvtryManagedLoc': 'production_invtry_managed_loc', 'availabilityCheckType': 'availability_check_type', 'fiscalYearVariant': 'fiscal_year_variant', 'profitCenter': 'profit_center', 'mrpType': 'mrp_type'}, 'product_storage_locations': {'product': 'product', 'plant': 'plant', 'storageLocation': 'storage_location', 'physicalInventoryBlockInd': 'physical_inventory_block_ind', 'dateOfLastPostedCntUnRstrcdStk': 'date_of_last_posted_cnt_un_rstrcd_stk'}, 'products': {'product': 'product', 'productType': 'product_type', 'crossPlantStatus': 'cross_plant_status', 'crossPlantStatusValidityDate': 'cross_plant_status_validity_date', 'creationDate': 'creation_date', 'createdByUser': 'created_by_user', 'lastChangeDate': 'last_change_date', 'lastChangeDateTime': 'last_change_datetime', 'isMarkedForDeletion': 'is_marked_for_deletion', 'productOldId': 'product_old_id', 'grossWeight': 'gross_weight', 'weightUnit': 'weight_unit', 'netWeight': 'net_weight', 'productGroup': 'product_group', 'baseUnit': 'base_unit', 'division': 'division', 'industrySector': 'industry_sector'}, 'sales_order_headers': {'salesOrder': 'sales_order', 'salesOrderType': 'sales_order_type', 'salesOrganization': 'sales_organization', 'distributionChannel': 'distribution_channel', 'organizationDivision': 'organization_division', 'salesGroup': 'sales_group', 'salesOffice': 'sales_office', 'soldToParty': 'sold_to_party', 'creationDate': 'creation_date', 'createdByUser': 'created_by_user', 'lastChangeDateTime': 'last_change_datetime', 'totalNetAmount': 'total_net_amount', 'overallDeliveryStatus': 'overall_delivery_status', 'overallOrdReltdBillgStatus': 'overall_ord_reltd_billg_status', 'overallSdDocReferenceStatus': 'overall_sd_doc_reference_status', 'transactionCurrency': 'transaction_currency', 'pricingDate': 'pricing_date', 'requestedDeliveryDate': 'requested_delivery_date', 'headerBillingBlockReason': 'header_billing_block_reason', 'deliveryBlockReason': 'delivery_block_reason', 'incotermsClassification': 'incoterms_classification', 'incotermsLocation1': 'incoterms_location1', 'customerPaymentTerms': 'customer_payment_terms', 'totalCreditCheckStatus': 'total_credit_check_status'}, 'sales_order_items': {'salesOrder': 'sales_order', 'salesOrderItem': 'sales_order_item', 'salesOrderItemCategory': 'sales_order_item_category', 'material': 'material', 'requestedQuantity': 'requested_quantity', 'requestedQuantityUnit': 'requested_quantity_unit', 'transactionCurrency': 'transaction_currency', 'netAmount': 'net_amount', 'materialGroup': 'material_group', 'productionPlant': 'production_plant', 'storageLocation': 'storage_location', 'salesDocumentRjcnReason': 'sales_document_rjcn_reason', 'itemBillingBlockReason': 'item_billing_block_reason'}, 'sales_order_schedule_lines': {'salesOrder': 'sales_order', 'salesOrderItem': 'sales_order_item', 'scheduleLine': 'schedule_line', 'confirmedDeliveryDate': 'confirmed_delivery_date', 'orderQuantityUnit': 'order_quantity_unit', 'confdOrderQtyByMatlAvailCheck': 'confd_order_qty_by_matl_avail_check'}}

NESTED_TIME_JSON_KEYS = {'billing_document_headers': ['creationTime'], 'billing_document_cancellations': ['creationTime'], 'business_partners': ['creationTime'], 'outbound_delivery_headers': ['creationTime', 'actualGoodsMovementTime']}

for table_name, col_map in TABLE_COLUMN_MAPS.items():
    path_pattern = os.path.join(DATA_DIR, table_name, '*.jsonl')
    files = glob.glob(path_pattern)
    if not files:
        print(f"WARNING: no files for {table_name}, skipping")
        continue

    nested_keys = NESTED_TIME_JSON_KEYS.get(table_name, [])
    # Only include non-nested keys in read_json columns spec
    readable_cols = {k: v for k, v in col_map.items() if k not in nested_keys}

    col_spec = ", ".join([f"'{k}': 'VARCHAR'" for k in readable_cols])
    
    # Build SELECT clause: cast each column to target name
    selects = []
    for json_key, sql_col in readable_cols.items():
        selects.append(f"NULLIF(TRIM(\"{json_key}\"), '') AS {sql_col}")
    
    select_clause = ",\n        ".join(selects)
    
    query = f"""
    INSERT INTO {table_name}
        ({", ".join(readable_cols.values())})
    SELECT
        {select_clause}
    FROM read_json_auto('{path_pattern}', columns={{{col_spec}}})
    """
    con.execute(query)
    print(f"Loaded {table_name}")

print("\nAll tables loaded.")


## Verification


In [ ]:
%%sql
SELECT table_name, estimated_size AS row_count
FROM duckdb_tables()
ORDER BY table_name;


In [ ]:
%%sql

SELECT * FROM sales_order_headers LIMIT 5;


## 1. Validation: DDL vs Notebook Schema

The notebook's inline DDL closely mirrors `ddl.sql` with minor type width differences:
- **Notebook DDL**: uses tight SAP-length `VARCHAR(10)`, `VARCHAR(4)`, `NUMERIC(15,2)` etc.
- **`ddl.sql`**: uses relaxed `VARCHAR(255)`, `NUMERIC(18,4)` for flexibility in PostgreSQL

Both define the same 19 tables with identical column names and composite PKs. The notebook DDL is fine for DuckDB analysis; `ddl.sql` is the canonical schema for PostgreSQL.

**Data loading**: The notebook loads data from `../data/sap-o2c-data/` JSONL files using explicit camelCase→snake_case mapping. Nested time objects (`{hours, minutes, seconds}`) are excluded from the initial load.

In [ ]:
## EDA: Null / Empty / Constant analysis per table
eda_results = {}
for tbl in con.execute("SELECT table_name FROM duckdb_tables() ORDER BY table_name").fetchdf()['table_name']:
    cols = con.execute(f"SELECT column_name FROM duckdb_columns() WHERE table_name = '{tbl}'").fetchdf()['column_name'].tolist()
    n = con.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    if n == 0:
        continue
    stats = []
    for c in cols:
        row = con.execute(f"""
            SELECT
                COUNT(*) FILTER (WHERE "{c}" IS NULL) AS nulls,
                COUNT(*) FILTER (WHERE CAST("{c}" AS VARCHAR) = '') AS empties,
                COUNT(DISTINCT "{c}") AS uniq
            FROM {tbl}
        """).fetchone()
        null_pct = round(row[0] / n * 100, 1)
        empty_pct = round(row[1] / n * 100, 1)
        missing_pct = null_pct + empty_pct
        stats.append({
            'column': c, 'nulls': row[0], 'null%': null_pct,
            'empties': row[1], 'empty%': empty_pct,
            'missing%': missing_pct, 'unique': row[2]
        })
    df = pd.DataFrame(stats)
    eda_results[tbl] = df
    issues = df[df['missing%'] > 50]
    if not issues.empty:
        print(f"\n{tbl} ({n} rows) — columns with >50% missing:")
        for _, r in issues.iterrows():
            print(f"  {r['column']}: {r['missing%']}% missing (null={r['null%']}%, empty={r['empty%']}%), {r['unique']} unique")

print("\n✓ EDA complete. Results stored in eda_results dict.")

In [ ]:
## Cross-table FK validation: check value overlap for key join columns
fk_checks = [
    ('sales_order_headers', 'sold_to_party', 'business_partners', 'customer'),
    ('sales_order_items', 'sales_order', 'sales_order_headers', 'sales_order'),
    ('outbound_delivery_items', 'reference_sd_document', 'sales_order_headers', 'sales_order'),
    ('outbound_delivery_items', 'delivery_document', 'outbound_delivery_headers', 'delivery_document'),
    ('billing_document_items', 'reference_sd_document', 'outbound_delivery_headers', 'delivery_document'),
    ('billing_document_items', 'billing_document', 'billing_document_headers', 'billing_document'),
    ('billing_document_headers', 'accounting_document', 'journal_entry_items_accounts_receivable', 'accounting_document'),
    ('journal_entry_items_accounts_receivable', 'accounting_document', 'payments_accounts_receivable', 'accounting_document'),
    ('billing_document_headers', 'cancelled_billing_document', 'billing_document_cancellations', 'billing_document'),
    ('sales_order_items', 'material', 'products', 'product'),
    ('billing_document_items', 'material', 'products', 'product'),
    ('product_plants', 'plant', 'plants', 'plant'),
    ('business_partner_addresses', 'business_partner', 'business_partners', 'business_partner'),
]

print(f"{'Source':<55} {'Target':<55} {'Overlap':>8} {'Src%':>6} {'Tgt%':>6}")
print("─" * 136)
for t1, c1, t2, c2 in fk_checks:
    row = con.execute(f"""
        WITH
          src AS (SELECT DISTINCT "{c1}" AS v FROM {t1} WHERE "{c1}" IS NOT NULL AND CAST("{c1}" AS VARCHAR) != ''),
          tgt AS (SELECT DISTINCT "{c2}" AS v FROM {t2} WHERE "{c2}" IS NOT NULL AND CAST("{c2}" AS VARCHAR) != ''),
          ovr AS (SELECT COUNT(*) AS n FROM src JOIN tgt ON src.v = tgt.v)
        SELECT
          (SELECT COUNT(*) FROM src) AS src_n,
          (SELECT COUNT(*) FROM tgt) AS tgt_n,
          (SELECT n FROM ovr) AS overlap
    """).fetchone()
    src_n, tgt_n, overlap = row
    src_pct = round(overlap / src_n * 100) if src_n else 0
    tgt_pct = round(overlap / tgt_n * 100) if tgt_n else 0
    print(f"{t1}.{c1:<40} {t2}.{c2:<40} {overlap:>8} {src_pct:>5}% {tgt_pct:>5}%")

## 2. EDA Summary (per table)

### Row Counts
| Table | Rows | Columns |
|-------|------|---------|
| billing_document_cancellations | 80 | 14 |
| billing_document_headers | 163 | 14 |
| billing_document_items | 245 | 9 |
| business_partner_addresses | 8 | 20 |
| business_partners | 8 | 19 |
| customer_company_assignments | 8 | 13 |
| customer_sales_area_assignments | 28 | 19 |
| journal_entry_items_accounts_receivable | 123 | 22 |
| outbound_delivery_headers | 86 | 13 |
| outbound_delivery_items | 137 | 11 |
| payments_accounts_receivable | 120 | 23 |
| plants | 44 | 14 |
| product_descriptions | 69 | 3 |
| product_plants | 3,036 | 9 |
| product_storage_locations | 16,723 | 5 |
| products | 69 | 17 |
| sales_order_headers | 100 | 24 |
| sales_order_items | 167 | 13 |
| sales_order_schedule_lines | 179 | 6 |

### Data Quality Issues

**Nested Objects (require TIME casting):**
- `billing_document_cancellations.creation_time` — `{hours, minutes, seconds}` object
- `billing_document_headers.creation_time` — same
- `business_partners.creation_time` — same
- `outbound_delivery_headers.creation_time` — same
- `outbound_delivery_headers.actual_goods_movement_time` — same

**High Missing (>80% null + empty):**

| Table | Column | Missing % | Notes |
|-------|--------|-----------|-------|
| billing_document_cancellations | cancelled_billing_document | 100% | All empty strings (cancellations ARE the cancel doc) |
| business_partner_addresses | po_box, po_box_* (6 cols), tax_jurisdiction, transport_zone | 100% | PO box fields unused |
| business_partners | first_name, last_name, correspondence_language, industry | 100% | Org partners only (category=2) |
| customer_company_assignments | accounting_clerk* (4 cols), alt_payer, payment_blocking/methods | 100% | Clerk fields unpopulated |
| customer_sales_area_assignments | billing_blocked, credit_control, sales_district/group/office, supplying_plant | 100% | Area-specific blocks unused |
| journal_entry_items_AR | assignment_reference, cost_center | 100% | |
| outbound_delivery_headers | actual_goods_movement_date (97%), delivery/billing block reason, proof_of_delivery | 97-100% | Most deliveries not yet moved |
| outbound_delivery_items | item_billing_block_reason, last_change_date | 100% | |
| payments_AR | assignment_reference, cost_center, invoice_reference, sales_document* | 100% | **Critical: invoice_reference is NULL — breaks O2C join** |
| plants | default_purchasing_org, plant_category | 100% | |
| product_plants | country/region_of_origin, fiscal_year_variant, production_invtry_managed_loc | 100% | |
| product_storage_locations | date_of_last_posted, physical_inventory_block_ind | 100% | |
| products | cross_plant_status, cross_plant_status_validity_date | 100% | |
| sales_order_headers | delivery/billing block, overall_billing/sd_ref_status, sales_group/office, credit_check | 100% | |
| sales_order_items | item_billing_block_reason (100%), sales_document_rjcn_reason (82%) | 82-100% | |

**Constant-Value Columns (single value across all rows):**
- `company_code` = 'ABCD' (all financial tables) — single-company dataset
- `fiscal_year` = '2025' — single-period dataset
- `transaction_currency` / `company_code_currency` = 'INR' — single-currency
- `sales_organization` = 'ABCD', `division` = '99', `distribution_channel` varies slightly
- Various status/type fields: `billing_document_type` = 'F2', `sales_order_type` = 'OR', etc.

## 3. Per-Table Relationship Analysis (PK, FK Candidates, Joins)

### Primary Key Analysis

| Table | PK | Type |
|-------|------|------|
| billing_document_headers | `billing_document` | Single |
| billing_document_cancellations | `billing_document` | Single |
| billing_document_items | `(billing_document, billing_document_item)` | Composite |
| business_partners | `business_partner` | Single |
| business_partner_addresses | `(business_partner, address_id)` | Composite |
| customer_company_assignments | `(customer, company_code)` | Composite |
| customer_sales_area_assignments | `(customer, sales_org, dist_channel, division)` | Composite |
| journal_entry_items_AR | `(company_code, fiscal_year, accounting_document, accounting_document_item)` | Composite |
| outbound_delivery_headers | `delivery_document` | Single |
| outbound_delivery_items | `(delivery_document, delivery_document_item)` | Composite |
| payments_AR | `(company_code, fiscal_year, accounting_document, accounting_document_item)` | Composite |
| plants | `plant` | Single |
| products | `product` | Single |
| product_descriptions | `(product, language)` | Composite |
| product_plants | `(product, plant)` | Composite |
| product_storage_locations | `(product, plant, storage_location)` | Composite |
| sales_order_headers | `sales_order` | Single |
| sales_order_items | `(sales_order, sales_order_item)` | Composite |
| sales_order_schedule_lines | `(sales_order, sales_order_item, schedule_line)` | Composite |

### Foreign Key Candidates (confirmed by value overlap)

#### Transactional O2C Chain
| Source Table | Source Column | Target Table | Target Column | Overlap |
|-------------|--------------|-------------|--------------|---------|
| sales_order_headers | `sold_to_party` | business_partners | `customer` | 100% → 100% |
| sales_order_items | `sales_order` | sales_order_headers | `sales_order` | 100% → 100% |
| sales_order_schedule_lines | `(sales_order, sales_order_item)` | sales_order_items | `(sales_order, sales_order_item)` | 100% → 100% |
| outbound_delivery_items | `reference_sd_document` | sales_order_headers | `sales_order` | 100% → 86% |
| outbound_delivery_items | `delivery_document` | outbound_delivery_headers | `delivery_document` | 100% → 100% |
| billing_document_items | `reference_sd_document` | outbound_delivery_headers | `delivery_document` | 100% → 97% |
| billing_document_items | `billing_document` | billing_document_headers | `billing_document` | 100% → 100% |
| billing_document_headers | `accounting_document` | journal_entry_items_AR | `accounting_document` | 75% → 100% |
| journal_entry_items_AR | `accounting_document` | payments_AR | `accounting_document` | 98% → 100% |
| billing_document_headers | `cancelled_billing_document` | billing_document_cancellations | `billing_document` | 100% → 100% |

#### Master Data
| Source Table | Source Column | Target Table | Target Column | Overlap |
|-------------|--------------|-------------|--------------|---------|
| business_partner_addresses | `business_partner` | business_partners | `business_partner` | 100% → 100% |
| customer_company_assignments | `customer` | business_partners | `customer` | 100% → 100% |
| customer_sales_area_assignments | `customer` | business_partners | `customer` | 100% → 100% |
| sales_order_items | `material` | products | `product` | 100% → 100% |
| billing_document_items | `material` | products | `product` | 100% → 80% |
| product_descriptions | `product` | products | `product` | 100% → 100% |
| product_plants | `product` | products | `product` | 100% → 100% |
| product_plants | `plant` | plants | `plant` | 100% → 100% |
| product_storage_locations | `product` | products | `product` | 100% → 100% |
| product_storage_locations | `plant` | plants | `plant` | 100% → 100% |
| sales_order_items | `production_plant` | plants | `plant` | 100% → 16% |
| outbound_delivery_items | `plant` | plants | `plant` | 100% → 11% |
| billing_document_headers | `sold_to_party` | business_partners | `customer` | 100% → 50% |

### Implicit Relationships (different column names)
- `sold_to_party` in sales/billing headers → `customer` in business_partners
- `material` in order/billing items → `product` in products
- `reference_sd_document` in delivery/billing items → `sales_order` or `delivery_document` (context-dependent)
- `accounting_document` bridges billing → journal entries → payments

## 4. Cleanup Plan

### Columns to DROP (100% missing, no analytical value)

```sql
-- billing_document_cancellations
ALTER TABLE billing_document_cancellations DROP COLUMN cancelled_billing_document;

-- business_partner_addresses (8 columns)
ALTER TABLE business_partner_addresses
  DROP COLUMN po_box,
  DROP COLUMN po_box_deviating_city_name,
  DROP COLUMN po_box_deviating_country,
  DROP COLUMN po_box_deviating_region,
  DROP COLUMN po_box_lobby_name,
  DROP COLUMN po_box_postal_code,
  DROP COLUMN tax_jurisdiction,
  DROP COLUMN transport_zone;

-- business_partners (4 columns — org-only dataset)
ALTER TABLE business_partners
  DROP COLUMN first_name,
  DROP COLUMN last_name,
  DROP COLUMN correspondence_language,
  DROP COLUMN industry;

-- customer_company_assignments (7 columns)
ALTER TABLE customer_company_assignments
  DROP COLUMN accounting_clerk,
  DROP COLUMN accounting_clerk_fax_number,
  DROP COLUMN accounting_clerk_internet_address,
  DROP COLUMN accounting_clerk_phone_number,
  DROP COLUMN alternative_payer_account,
  DROP COLUMN payment_blocking_reason,
  DROP COLUMN payment_methods_list;

-- customer_sales_area_assignments (6 columns)
ALTER TABLE customer_sales_area_assignments
  DROP COLUMN billing_is_blocked_for_customer,
  DROP COLUMN credit_control_area,
  DROP COLUMN sales_district,
  DROP COLUMN sales_group,
  DROP COLUMN sales_office,
  DROP COLUMN supplying_plant;

-- journal_entry_items_accounts_receivable (2 columns)
ALTER TABLE journal_entry_items_accounts_receivable
  DROP COLUMN assignment_reference,
  DROP COLUMN cost_center;

-- outbound_delivery_headers (3 columns)
ALTER TABLE outbound_delivery_headers
  DROP COLUMN delivery_block_reason,
  DROP COLUMN header_billing_block_reason,
  DROP COLUMN overall_proof_of_delivery_status;

-- outbound_delivery_items (2 columns)
ALTER TABLE outbound_delivery_items
  DROP COLUMN item_billing_block_reason,
  DROP COLUMN last_change_date;

-- payments_accounts_receivable (6 columns — critical gap noted)
ALTER TABLE payments_accounts_receivable
  DROP COLUMN assignment_reference,
  DROP COLUMN cost_center,
  DROP COLUMN invoice_reference,           -- 100% NULL; needs upstream fix
  DROP COLUMN invoice_reference_fiscal_year,
  DROP COLUMN sales_document,
  DROP COLUMN sales_document_item;

-- plants (2 columns)
ALTER TABLE plants
  DROP COLUMN default_purchasing_organization,
  DROP COLUMN plant_category;

-- product_plants (4 columns)
ALTER TABLE product_plants
  DROP COLUMN country_of_origin,
  DROP COLUMN region_of_origin,
  DROP COLUMN production_invtry_managed_loc,
  DROP COLUMN fiscal_year_variant;

-- product_storage_locations (2 columns)
ALTER TABLE product_storage_locations
  DROP COLUMN physical_inventory_block_ind,
  DROP COLUMN date_of_last_posted_cnt_un_rstrcd_stk;

-- products (2 columns)
ALTER TABLE products
  DROP COLUMN cross_plant_status,
  DROP COLUMN cross_plant_status_validity_date;

-- sales_order_headers (7 columns)
ALTER TABLE sales_order_headers
  DROP COLUMN delivery_block_reason,
  DROP COLUMN header_billing_block_reason,
  DROP COLUMN overall_ord_reltd_billg_status,
  DROP COLUMN overall_sd_doc_reference_status,
  DROP COLUMN sales_group,
  DROP COLUMN sales_office,
  DROP COLUMN total_credit_check_status;

-- sales_order_items (1 column)
ALTER TABLE sales_order_items
  DROP COLUMN item_billing_block_reason;
```

**Total: ~56 columns removed across 15 tables.**

### Constant-Value Columns (KEEP but note)
Columns like `company_code`, `fiscal_year`, `transaction_currency` are constant in this dataset but structurally important for multi-company/multi-year scenarios. **Keep in schema, but don't use for filtering or graph edges.**

## 5. Improved Schema with Explicit Foreign Keys

### Design Decisions
1. **Keep SAP table names** — team already familiar with them; renaming adds confusion
2. **Add explicit FK constraints** — enables graph traversal and referential integrity
3. **Drop 56 empty columns** — less noise for queries and graph exploration
4. **Keep constant columns** — structurally important for multi-tenant expansion

### FK Constraints to ADD to `ddl.sql`

```sql
-- ═══════════════════════════════════════════════════════════════
-- TRANSACTIONAL O2C CHAIN
-- ═══════════════════════════════════════════════════════════════

-- Sales Order → Customer (via sold_to_party → business_partners.customer)
ALTER TABLE sales_order_headers
  ADD CONSTRAINT fk_so_customer
  FOREIGN KEY (sold_to_party) REFERENCES business_partners(customer);

-- Sales Order Items → Sales Order Header
ALTER TABLE sales_order_items
  ADD CONSTRAINT fk_soi_so
  FOREIGN KEY (sales_order) REFERENCES sales_order_headers(sales_order);

-- Sales Order Schedule Lines → Sales Order Items
ALTER TABLE sales_order_schedule_lines
  ADD CONSTRAINT fk_sosl_soi
  FOREIGN KEY (sales_order, sales_order_item)
  REFERENCES sales_order_items(sales_order, sales_order_item);

-- Outbound Delivery Items → Sales Order (via reference_sd_document)
ALTER TABLE outbound_delivery_items
  ADD CONSTRAINT fk_odi_so
  FOREIGN KEY (reference_sd_document) REFERENCES sales_order_headers(sales_order);

-- Outbound Delivery Items → Outbound Delivery Header
ALTER TABLE outbound_delivery_items
  ADD CONSTRAINT fk_odi_odh
  FOREIGN KEY (delivery_document) REFERENCES outbound_delivery_headers(delivery_document);

-- Billing Document Items → Billing Document Header
ALTER TABLE billing_document_items
  ADD CONSTRAINT fk_bdi_bdh
  FOREIGN KEY (billing_document) REFERENCES billing_document_headers(billing_document);

-- Billing Document Items → Outbound Delivery (via reference_sd_document)
ALTER TABLE billing_document_items
  ADD CONSTRAINT fk_bdi_delivery
  FOREIGN KEY (reference_sd_document) REFERENCES outbound_delivery_headers(delivery_document);

-- Billing Document Header → Customer
ALTER TABLE billing_document_headers
  ADD CONSTRAINT fk_bdh_customer
  FOREIGN KEY (sold_to_party) REFERENCES business_partners(customer);

-- Billing Document Header → Cancellation (self-referential link)
ALTER TABLE billing_document_headers
  ADD CONSTRAINT fk_bdh_cancellation
  FOREIGN KEY (cancelled_billing_document) REFERENCES billing_document_cancellations(billing_document);

-- ═══════════════════════════════════════════════════════════════
-- FINANCIAL CHAIN (via accounting_document)
-- ═══════════════════════════════════════════════════════════════

-- Note: journal_entry_items_AR and payments_AR use composite PKs.
-- The link from billing_document_headers.accounting_document to these tables
-- is a logical FK but cannot be a simple REFERENCES due to composite PK mismatch.
-- Use VIEW or application-level join instead:
--   billing_document_headers.accounting_document = journal_entry_items_AR.accounting_document
--   journal_entry_items_AR.accounting_document = payments_AR.accounting_document

-- Journal Entry → Customer
ALTER TABLE journal_entry_items_accounts_receivable
  ADD CONSTRAINT fk_je_customer
  FOREIGN KEY (customer) REFERENCES business_partners(customer);

-- Payment → Customer
ALTER TABLE payments_accounts_receivable
  ADD CONSTRAINT fk_pay_customer
  FOREIGN KEY (customer) REFERENCES business_partners(customer);

-- ═══════════════════════════════════════════════════════════════
-- MASTER DATA
-- ═══════════════════════════════════════════════════════════════

-- Business Partner Address → Business Partner
ALTER TABLE business_partner_addresses
  ADD CONSTRAINT fk_bpa_bp
  FOREIGN KEY (business_partner) REFERENCES business_partners(business_partner);

-- Customer Company → Business Partner (via customer)
ALTER TABLE customer_company_assignments
  ADD CONSTRAINT fk_cca_bp
  FOREIGN KEY (customer) REFERENCES business_partners(customer);

-- Customer Sales Area → Business Partner (via customer)
ALTER TABLE customer_sales_area_assignments
  ADD CONSTRAINT fk_csaa_bp
  FOREIGN KEY (customer) REFERENCES business_partners(customer);

-- Product Descriptions → Product
ALTER TABLE product_descriptions
  ADD CONSTRAINT fk_pd_product
  FOREIGN KEY (product) REFERENCES products(product);

-- Product Plants → Product + Plant
ALTER TABLE product_plants
  ADD CONSTRAINT fk_pp_product
  FOREIGN KEY (product) REFERENCES products(product);
ALTER TABLE product_plants
  ADD CONSTRAINT fk_pp_plant
  FOREIGN KEY (plant) REFERENCES plants(plant);

-- Product Storage Locations → Product + Plant
ALTER TABLE product_storage_locations
  ADD CONSTRAINT fk_psl_product
  FOREIGN KEY (product) REFERENCES products(product);
ALTER TABLE product_storage_locations
  ADD CONSTRAINT fk_psl_plant
  FOREIGN KEY (plant) REFERENCES plants(plant);

-- Sales Order Items → Product (via material)
ALTER TABLE sales_order_items
  ADD CONSTRAINT fk_soi_product
  FOREIGN KEY (material) REFERENCES products(product);

-- Sales Order Items → Plant (via production_plant)
ALTER TABLE sales_order_items
  ADD CONSTRAINT fk_soi_plant
  FOREIGN KEY (production_plant) REFERENCES plants(plant);

-- Outbound Delivery Items → Plant
ALTER TABLE outbound_delivery_items
  ADD CONSTRAINT fk_odi_plant
  FOREIGN KEY (plant) REFERENCES plants(plant);

-- Billing Document Items → Product (via material)
ALTER TABLE billing_document_items
  ADD CONSTRAINT fk_bdi_product
  FOREIGN KEY (material) REFERENCES products(product);
```

### Unique Index Additions (for FK targets)
```sql
-- business_partners.customer is used as FK target but is not the PK
CREATE UNIQUE INDEX uq_bp_customer ON business_partners(customer);
```

## 6. End-to-End Relationship Graph (for graph traversal)

### Full O2C Flow (primary traversal path)

```
┌──────────────┐     sold_to_party     ┌─────────────────┐
│  Business    │◄─────────────────────│  Sales Order     │
│  Partners    │     = customer        │  Headers         │
└──────┬───────┘                       └────────┬────────┘
       │                                        │ sales_order
       │ business_partner                       ▼
┌──────▼───────┐                       ┌─────────────────┐
│  BP          │                       │  Sales Order     │
│  Addresses   │                       │  Items           │──── material ────► Products
└──────────────┘                       └────────┬────────┘       │
       │                                        │                 ▼
┌──────▼───────┐                       ┌────────▼────────┐  ┌──────────┐
│  Customer    │                       │  Schedule Lines  │  │  Plants  │
│  Company     │                       └─────────────────┘  └──────────┘
│  Assignments │                                                  ▲
└──────────────┘            reference_sd_document = sales_order   │
       │                              ┌─────────────────┐         │
┌──────▼───────┐                      │  Outbound       │── plant ┘
│  Customer    │                      │  Delivery Items  │
│  Sales Area  │                      └────────┬────────┘
└──────────────┘                               │ delivery_document
                                       ┌───────▼────────┐
                                       │  Outbound       │
                                       │  Delivery Hdrs  │
                                       └────────┬────────┘
                              ref_sd_doc = delivery_document
                                       ┌────────▼────────┐
                                       │  Billing Doc    │
                                       │  Items          │── material ──► Products
                                       └────────┬────────┘
                                                │ billing_document
                                       ┌────────▼────────┐   cancelled_billing_doc
                                       │  Billing Doc    │─────────────────────────┐
                                       │  Headers        │                         │
                                       └───┬─────────┬───┘                         ▼
                          accounting_doc   │         │ sold_to_party    ┌───────────────────┐
                                    ┌──────▼──┐      │                 │  Billing Doc      │
                                    │ Journal │      └──► BP           │  Cancellations    │
                                    │ Entry   │                        └───────────────────┘
                                    │ Items AR│
                                    └────┬────┘
                          accounting_doc │
                                    ┌────▼────┐
                                    │Payments │
                                    │   AR    │
                                    └─────────┘
```

### Key Traversal Paths for Graph API

| Path Name | Hops | Join Chain |
|-----------|------|------------|
| **Customer → Orders** | 1 | `business_partners.customer = sales_order_headers.sold_to_party` |
| **Order → Deliveries** | 2 | `SO → SO_items.sales_order` then `SO_items ← delivery_items.reference_sd_document` |
| **Order → Invoices** | 4 | `SO → delivery_items → delivery_headers → billing_items.reference_sd_document → billing_headers` |
| **Invoice → Payment** | 2 | `billing_headers.accounting_document = journal_entries.accounting_document = payments.accounting_document` |
| **Full O2C** | 7 | Customer → SO → Delivery → Invoice → JE → Payment |
| **Product Trace** | 3 | `products → product_plants → plants` + `product_descriptions` |
| **Cancellation Check** | 1 | `billing_headers.cancelled_billing_document = cancellations.billing_document` |

### Critical Data Gap

**`payments_accounts_receivable.invoice_reference` is 100% NULL.** This means the direct billing→payment link via invoice number is broken. The alternative join path uses `accounting_document` (98-100% overlap between journal entries and payments), which works but is an indirect path through the financial posting layer rather than a direct O2C commercial link.

**Recommendation:** Investigate upstream data extraction to populate `invoice_reference`. Until then, use the `accounting_document` bridge for payment tracing.